In [ ]:
#compute optimal thershold age 12

In [1]:
import os
import sys
import json
import copy
import socket
import getpass
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
sys.path.append('../')
import pickle
import warnings
from sklearn.metrics import recall_score, matthews_corrcoef, roc_auc_score, f1_score
from collections import defaultdict
import json

warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

def flatten(l):
    return [item for sublist in l for item in sublist]

def compute_opt_thres(target, pred):
    opt_thres = 0
    opt_f1 = 0
    for i in np.arange(0.001, 0.999, 0.001):
        f1 = f1_score(target, pred >= i)
        if f1 >= opt_f1:
            opt_thres = i
            opt_f1 = f1
    return opt_thres

plt.rcParams.update({'font.size': 20})

In [2]:
#root_dir = Path(f'/path/to/your/root')
root_dir = Path(f'/home/lchanch/initial_training/output_sweep_12/grid_age_mimic_12')

In [3]:
if Path('opt_thres.pkl').is_file():
    already_computed = set(pickle.load(Path('opt_thres.pkl').open('rb')).keys())
else:
    already_computed = set()

In [4]:
results = []
for i in tqdm(root_dir.glob('**/done')):
    args = json.load((i.parent/'args.json').open('r'))
    if (args['dataset'][0], args['task'], args['attr'], args['algorithm']) in already_computed:
        continue
    
    final_res = pickle.load((i.parent/'final_results.pkl').open('rb'))
    
    ssets = ['va', 'te', 'MIMIC-sex-te', 'CheXpert-sex-te']
  
    for sset in ssets:
        if sset in final_res:
            args[f'{sset}_auroc'] = final_res[sset]['overall']['AUROC']
            if sset == 'va':
                args[f'{sset}_min_attr_auroc'] = final_res[sset]['min_attr']['AUROC']
    args['va_y'] = final_res['va']['y']
    args['va_preds'] = final_res['va']['preds']
    
    results.append(args)
df = pd.DataFrame(results)

136it [00:00, 671.88it/s]


In [5]:
df['dataset'] = df['dataset'].apply(lambda x: x[0])

In [6]:
df.shape

(136, 33)

In [7]:
df.head()

,store_name,dataset,task,attr,group_def,algorithm,output_dir,data_dir,hparams,hparams_seed,...,checkpoint_freq,skip_model_save,debug,image_arch,aug,va_auroc,va_min_attr_auroc,te_auroc,va_y,va_preds
0,c2e4efad973092323dcfa0bbcbdbb212,MIMIC,Pleural Effusion,age,group,ERM,/home/lchanch/initial_training/output_sweep_12...,/home/lchanch/initial_training/image_df,{},0,...,1000,False,False,densenet_sup_in1k,basic2,0.886403,0.858946,0.886671,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.087198734, 0.015593688, 0.014737086, 0.0105..."
1,c88afe7356e092efc0fe5343b4196b8a,MIMIC,Cardiomegaly,age,group,ERM,/home/lchanch/initial_training/output_sweep_12...,/home/lchanch/initial_training/image_df,{},4,...,1000,False,False,densenet_sup_in1k,basic2,0.574789,0.546319,0.568069,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, ...","[0.20267992, 0.20256591, 0.2032918, 0.2019469,..."
2,c0e27393a407076be5bd0d11521839be,MIMIC,Pleural Effusion,age,group,ERM,/home/lchanch/initial_training/output_sweep_12...,/home/lchanch/initial_training/image_df,{},5,...,1000,False,False,densenet_sup_in1k,basic2,0.919060,0.893322,0.922294,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.13837683, 0.029171545, 0.19524491, 0.053574..."
3,7e52cb5bd96dc57de34403a7d298cd1d,MIMIC,No Finding,age,group,ERM,/home/lchanch/initial_training/output_sweep_12...,/home/lchanch/initial_training/image_df,{},8,...,1000,False,False,densenet_sup_in1k,basic2,0.810780,0.747764,0.811363,"[0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ...","[0.31870314, 0.41006118, 0.52063066, 0.4705682..."
4,12dadaf66c3a13d465f2cc07b0272ebd,MIMIC,No Finding,age,group,ERM,/home/lchanch/initial_training/output_sweep_12...,/home/lchanch/initial_training/image_df,{},7,...,1000,False,False,densenet_sup_in1k,basic2,0.834656,0.764867,0.838164,"[0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ...","[0.23040779, 0.48901895, 0.29172546, 0.4186079..."


In [ ]:
#optimal_thresholds

In [ ]:
#best models según va_min_attr_auroc

In [8]:
best_models = df.groupby(['dataset', 'task', 'attr', 'algorithm']).apply(lambda x: x.loc[x['va_min_attr_auroc'].idxmax()])

/tmp/ipykernel_2207075/3613221529.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  best_models = df.groupby(['dataset', 'task', 'attr', 'algorithm']).apply(lambda x: x.loc[x['va_min_attr_auroc'].idxmax()])


In [9]:
best_models

store_name  \
dataset task             attr algorithm                                     
MIMIC   Cardiomegaly     age  ERM        1ee839af3a66af2f63fe2f628d1ccdab   
        No Finding       age  ERM        96f9f3456d6008db8711d40a2be36a50   
        Pleural Effusion age  ERM        89d0fa5c1bb5fd12170687b40fed2f18   
        Pneumothorax     age  ERM        19162059f762f34f95a00aebe9a887c1   

                                        dataset              task attr  \
dataset task             attr algorithm                                  
MIMIC   Cardiomegaly     age  ERM         MIMIC      Cardiomegaly  age   
        No Finding       age  ERM         MIMIC        No Finding  age   
        Pleural Effusion age  ERM         MIMIC  Pleural Effusion  age   
        Pneumothorax     age  ERM         MIMIC      Pneumothorax  age   

                                        group_def algorithm  \
dataset task             attr algorithm                       
MIMIC   Cardiomegaly     age  ERM           group       ERM   
        No Finding       age  ERM           group       ERM   
        Pleural Effusion age  ERM           group       ERM   
        Pneumothorax     age  ERM           group       ERM   

                                                                                output_dir  \
dataset task             attr algorithm                                                      
MIMIC   Cardiomegaly     age  ERM        /home/lchanch/initial_training/output_sweep_12...   
        No Finding       age  ERM        /home/lchanch/initial_training/output_sweep_12...   
        Pleural Effusion age  ERM        /home/lchanch/initial_training/output_sweep_12...   
        Pneumothorax     age  ERM        /home/lchanch/initial_training/output_sweep_12...   

                                                                        data_dir  \
dataset task             attr algorithm                                            
MIMIC   Cardiomegaly     age  ERM        /home/lchanch/initial_training/image_df   
        No Finding       age  ERM        /home/lchanch/initial_training/image_df   
        Pleural Effusion age  ERM        /home/lchanch/initial_training/image_df   
        Pneumothorax     age  ERM        /home/lchanch/initial_training/image_df   

                                        hparams  hparams_seed  ...  \
dataset task             attr algorithm                        ...   
MIMIC   Cardiomegaly     age  ERM            {}             1  ...   
        No Finding       age  ERM            {}             9  ...   
        Pleural Effusion age  ERM            {}             1  ...   
        Pneumothorax     age  ERM            {}             1  ...   

                                         checkpoint_freq skip_model_save  \
dataset task             attr algorithm                                    
MIMIC   Cardiomegaly     age  ERM                   1000           False   
        No Finding       age  ERM                   1000           False   
        Pleural Effusion age  ERM                   1000           False   
        Pneumothorax     age  ERM                   1000           False   

                                         debug         image_arch     aug  \
dataset task             attr algorithm                                     
MIMIC   Cardiomegaly     age  ERM        False  densenet_sup_in1k  basic2   
        No Finding       age  ERM        False  densenet_sup_in1k  basic2   
        Pleural Effusion age  ERM        False  densenet_sup_in1k  basic2   
        Pneumothorax     age  ERM        False  densenet_sup_in1k  basic2   

                                         va_auroc va_min_attr_auroc  te_auroc  \
dataset task             attr algorithm                                         
MIMIC   Cardiomegaly     age  ERM        0.815855          0.760496  0.810514   
        No Finding       age  ERM        0.843417          0.781130  0.846590   
        Pleural Effusion age  ERM        

In [10]:
opt_thres = {}
for idx, row in tqdm(best_models.iterrows(), total = len(best_models)):
    dataset, task, attr, algorithm = idx
#     if dataset not in opt_thres:
#         opt_thres[dataset] = {}
    opt_thres[(dataset, task, attr, algorithm)] = np.round(compute_opt_thres(row['va_y'], row['va_preds']), 3)

100%|██████████| 4/4 [00:20<00:00,  5.24s/it]


In [11]:
if Path('opt_thres.pkl').is_file():
    old_file = pickle.load(Path('opt_thres.pkl').open('rb'))
else:
    old_file = {}

In [12]:
opt_thres = {**old_file, **opt_thres}

In [13]:
opt_thres

{('MIMIC', 'Cardiomegaly', 'age', 'ERM'): 0.245,
 ('MIMIC', 'No Finding', 'age', 'ERM'): 0.403,
 ('MIMIC', 'Pleural Effusion', 'age', 'ERM'): 0.345,
 ('MIMIC', 'Pneumothorax', 'age', 'ERM'): 0.17}

In [14]:
#se observa que los opt thersholds son los mismos excepto para No Finding, que en el de 3 semillas era 0.427

In [15]:
pickle.dump(opt_thres, open('opt_thres_age_mimic_12.pkl', 'wb'))

In [16]:
best_models['hparams_seed']

dataset  task              attr  algorithm
MIMIC    Cardiomegaly      age   ERM          1
         No Finding        age   ERM          9
         Pleural Effusion  age   ERM          1
         Pneumothorax      age   ERM          1
Name: hparams_seed, dtype: int64

In [17]:
best_models[['store_name','hparams_seed']]

store_name  \
dataset task             attr algorithm                                     
MIMIC   Cardiomegaly     age  ERM        1ee839af3a66af2f63fe2f628d1ccdab   
        No Finding       age  ERM        96f9f3456d6008db8711d40a2be36a50   
        Pleural Effusion age  ERM        89d0fa5c1bb5fd12170687b40fed2f18   
        Pneumothorax     age  ERM        19162059f762f34f95a00aebe9a887c1   

                                         hparams_seed  
dataset task             attr algorithm                
MIMIC   Cardiomegaly     age  ERM                   1  
        No Finding       age  ERM                   9  
        Pleural Effusion age  ERM                   1  
        Pneumothorax     age  ERM                   1